**Chapter 15 – Processing Sequences Using RNNs and CNNs**

_This notebook contains all the sample code and solutions to the exercises in chapter 15._

Quelle: Géron, Aurélien. Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, O'Reilly Media, Incorporated, 2022. ProQuest Ebook Central, https://ebookcentral.proquest.com/lib/koln/detail.action?docID=30168989.

https://github.com/ageron/handson-ml3/blob/main/15_processing_sequences_using_rnns_and_cnns.ipynb

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_07/01_sequences_rnns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_07/01_sequences_rnns.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.7 or above:

In [ ]:
import sys

assert sys.version_info >= (3, 7)

And TensorFlow ≥ 2.8:

In [ ]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

As we did in earlier chapters, let's define the default font sizes to make the figures prettier:

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

And let's create the `images/rnn` folder (if it doesn't already exist), and define the `save_fig()` function which is used through this notebook to save the figures in high-res for the book:

In [ ]:
from pathlib import Path

IMAGES_PATH = Path() / "images" / "rnn"
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = IMAGES_PATH / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

This chapter can be very slow without a GPU, so let's make sure there's one, or else issue a warning:

In [ ]:
if not tf.config.list_physical_devices('GPU'):
    print("No GPU was detected. Neural nets can be very slow without a GPU.")
    if "google.colab" in sys.modules:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if "kaggle_secrets" in sys.modules:
        print("Go to Settings > Accelerator and select GPU.")

# Basic RNNs

Let's download the ridership data from the ageron/data project. It originally comes from Chicago's Transit Authority, and was downloaded from the [Chicago's Data Portal](https://homl.info/ridership).

**Warning**: in recent Keras versions, `get_file()` now wraps the extracted directory inside a directory whose name ends with `_extracted`, so the following code checks for that:

In [ ]:
filepath = tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir=".",
    extract=True
)
if "_extracted" in filepath:
    ridership_path = Path(filepath) / "ridership"
else:
    ridership_path = Path(filepath).with_name("ridership")

In [ ]:
import pandas as pd
from pathlib import Path

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [ ]:
df.head()

Let's look at the first few months of 2019 (note that Pandas treats the range boundaries as inclusive):

In [ ]:
import matplotlib.pyplot as plt

df["2019-03":"2019-05"].plot(grid=True, marker=".", figsize=(8, 3.5))
save_fig("daily_ridership_plot")  # extra code – saves the figure for the book
plt.show()

In [ ]:
def to_windows(dataset, length):
    dataset = dataset.window(length, shift=1, drop_remainder=True)
    return dataset.flat_map(lambda window_ds: window_ds.batch(length))

Before we continue looking at the data, let's split the time series into three periods, for training, validation and testing. We won't look at the test data for now:

In [ ]:
rail_train = df["rail"]["2016-01":"2018-12"] / 1e6
rail_valid = df["rail"]["2019-01":"2019-05"] / 1e6
rail_test = df["rail"]["2019-06":] / 1e6

In [ ]:
seq_length = 56
tf.random.set_seed(42)  # extra code – ensures reproducibility
train_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_train.to_numpy(),
    targets=rail_train[seq_length:],
    sequence_length=seq_length,
    batch_size=32,
    shuffle=True,
    seed=42
)
valid_ds = tf.keras.utils.timeseries_dataset_from_array(
    rail_valid.to_numpy(),
    targets=rail_valid[seq_length:],
    sequence_length=seq_length,
    batch_size=32
)

In [ ]:
for X_sample, y_sample in train_ds.take(1):
    break

plt.figure(figsize=(8, 3.5))
plt.plot(range(seq_length), X_sample[0], "-o", label="Input sequence")
plt.plot(seq_length, y_sample[0], "rx", markersize=10, label="Target label")
plt.grid(True)
plt.legend()
plt.title("One Sample from train_ds and its Label")
plt.xlabel("Time Step")
plt.ylabel("Normalized Rail Ridership")
plt.show()

## Using a Simple RNN

In [ ]:
# extra code – defines a utility function we'll reuse several time

def fit_and_evaluate(model, train_set, valid_set, learning_rate, epochs=100):
    early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor="val_mae", patience=50, restore_best_weights=True)
    opt = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=opt, metrics=["mae"])
    history = model.fit(train_set, validation_data=valid_set, epochs=epochs,
                        callbacks=[early_stopping_cb])
    valid_loss, valid_mae = model.evaluate(valid_set)
    return valid_mae * 1e6

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility
univar_model = tf.keras.Sequential([
    tf.keras.layers.InputLayer(input_shape=[None, 1]), # Use InputLayer for explicit input shape
    tf.keras.layers.SimpleRNN(32),   # , return_sequences=True
    tf.keras.layers.Dense(1)  # no activation function by default
])

In [ ]:
# Define a path to save the model
univar_model_path = Path("univar_rnn_model_s2v.keras")

# Check if the model already exists
if univar_model_path.is_file():
    print("Loading pre-trained univar_model...")
    univar_model = tf.keras.models.load_model(univar_model_path)
else:
    print("Training univar_model...")
    # Train and evaluate the model
    fit_and_evaluate(univar_model, train_ds, valid_ds, learning_rate=0.05)
    # Save the trained model
    univar_model.save(univar_model_path)
    print("univar_model trained and saved.")

In [ ]:
for X_sample, y_sample in valid_ds.take(1):
    break

# Expand dimensions of X_sample to match model's expected input shape (batch_size, seq_length, 1)
X_sample_reshaped = tf.expand_dims(X_sample, axis=-1)
# print(X_sample_reshaped)

y_pred_rnn = univar_model.predict(X_sample_reshaped)
# print(y_pred_rnn)

ibatch = 12

plt.figure(figsize=(8, 3.5))
plt.plot(range(seq_length), X_sample[ibatch], "-o", label="Input sequence")
plt.plot(seq_length, y_sample[ibatch], "rx", markersize=10, label="True target")
plt.plot(seq_length, y_pred_rnn[ibatch, 0], "gx", markersize=10, label="RNN prediction")
plt.grid(True)
plt.legend()
plt.title("RNN Prediction for a Validation Sample")
plt.xlabel("Time Step")
plt.ylabel("Normalized Rail Ridership")
plt.show()

## Forecasting Several Steps Ahead

In [ ]:
import numpy as np

X = rail_valid.to_numpy()[np.newaxis, :seq_length, np.newaxis]
for step_ahead in range(14):
    y_pred_one = univar_model.predict(X)
    X = np.concatenate([X, y_pred_one.reshape(1, 1, 1)], axis=1)

In [ ]:
# extra code – generates and saves Figure 15–11

# The forecasts start on 2019-02-26, as it is the 57th day of 2019, and they end
# on 2019-03-11. That's 14 days in total.
Y_pred = pd.Series(X[0, -14:, 0],
                   index=pd.date_range("2019-02-26", "2019-03-11"))

fig, ax = plt.subplots(figsize=(8, 3.5))
(rail_valid * 1e6)["2019-02-01":"2019-03-11"].plot(
    label="True", marker=".", ax=ax)
(Y_pred * 1e6).plot(
    label="Predictions", grid=True, marker="x", color="r", ax=ax)
ax.vlines("2019-02-25", 0, 1e6, color="k", linestyle="--", label="Today")
ax.set_ylim([200_000, 800_000])
plt.legend(loc="center left")
save_fig("forecast_ahead_plot")
plt.show()

# LSTMs

In [ ]:
tf.random.set_seed(42)  # extra code – ensures reproducibility - here sequence to sequence is used, as return_sequences = True
lstm_model = tf.keras.models.Sequential([
    tf.keras.layers.LSTM(32, return_sequences=True, input_shape=[None, 1]),
    tf.keras.layers.Dense(1)
])

In [ ]:
# Define a path to save the LSTM model
lstm_model_path = Path("lstm_model.keras")

# Check if the model already exists
if lstm_model_path.is_file():
    print("Loading pre-trained lstm_model...")
    lstm_model = tf.keras.models.load_model(lstm_model_path)
else:
    print("Training lstm_model...")
    # Train and evaluate the model
    fit_and_evaluate(lstm_model, train_ds, valid_ds,
                     learning_rate=0.1, epochs=25)
    # Save the trained model
    lstm_model.save(lstm_model_path)
    print("lstm_model trained and saved.")

Just training for 5 epochs to show that it works (you can increase this if you want):

In [ ]:
for X_sample, y_sample in valid_ds.take(1):
    break

y_pred_lstm = lstm_model.predict(X_sample)

plt.figure(figsize=(8, 3.5))
plt.plot(range(seq_length), X_sample[0], "-o", label="Input sequence")
plt.plot(seq_length, y_sample[0], "rx", markersize=10, label="True target")
plt.plot(seq_length, y_pred_lstm[0, -1, 0], "gx", markersize=10, label="LSTM prediction") # Last element of the sequence for prediction
plt.grid(True)
plt.legend()
plt.title("LSTM Prediction for a Validation Sample")
plt.xlabel("Time Step")
plt.ylabel("Normalized Rail Ridership")
plt.show()